> **2026-04-19 CONVENTION WARNING.** This notebook uses the older **UTC-midnight snap convention** (`snap_time = close_ts.floor('D') - N days`). Ship convention per `CLAUDE.md` is **ET-midnight** (`midnight ET on close−N`). The tier-2 Ridge model fit here is the structural basis for ridge_t2 (the ship candidate), but the timestamp / snap conventions were refit under ET in `notebooks/proposed_ship_stack_test.ipynb`. **Do not copy snap/phase timestamp logic from this notebook into new code** — use the ET convention documented in `CLAUDE.md` and `plans/plan_ridge_integration.md` §3.7.

# Ridge Tier-2: finite-pool features

**Intent:** add critic-pool composition features on top of tier 1 baseline. Does knowing **who is left to review** help Ridge predict phase-1 volume?

**Tier 2 features (3 new):**

| feature | formula |
|---|---|
| `remaining_base_rate_sum` | Σ_c base_rate[c] for c in A1_pool, c ∉ observed |
| `pool_mass_consumed`      | observed_base_rate_sum / total_base_rate_sum |
| `observed_top_tier_frac`  | \|observed ∩ A1_top_30\| / 30 |

**Base_rate source:** A1-recency pool. For each target, take the 20 most recent resolved movies before target close (standard `default_training_slugs`); compute per-critic base_rate = movies_reviewed_in_A1 / 20. LOO-clean — target's own reviews never contribute.

**Top-tier definition:** top 30 critics by base_rate within the A1 pool. Target-adaptive — matches the pool definition. These are the A1 workhorses: who reviewed the most of the 20 most-recent movies.

**Base stack:** tier-1 D (standardization + 14 features + α CV) from `phase1_ridge_tier1.pkl`.

**Plan doc:** `brainstorm/brainstorm_ridge_optimization.md` (§ Tier 2).

**Followup noted for later:** Jake wants an A3-gap-only-weighted variant tested once tier 2 lands (no Jaccard; use combined_score with α=1.0 weights on the 20 top-gap-matched movies). Captures embargo-anchor similarity without per-snap critic-overlap dependence.


In [ ]:
import sys
from pathlib import Path

NB_DIR = Path.cwd()
if NB_DIR.name != 'notebooks':
    NB_DIR = NB_DIR / 'notebooks'
if str(NB_DIR) not in sys.path:
    sys.path.insert(0, str(NB_DIR))

import pickle
import time

import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold

import _helpers as H

print(f'cohort: {len(H.close_date_map)} movies')


## Noon-shift setup (match tier 1 + three-way convention)

Mutate `H.reviews` to noon-shift day-level timestamps, and recompute `H.first_review_ts`, `H.gap_lookup` from the shifted copy. This makes `H.snapshot_state` produce observed-critic sets consistent with the tier-1 base.


In [ ]:
_day_mask = H.reviews['timestamp_confidence'] == 'd'
_n_shifted = int(_day_mask.sum())
H.reviews.loc[_day_mask, 'estimated_timestamp'] = (
    H.reviews.loc[_day_mask, 'estimated_timestamp'] + pd.Timedelta(hours=12)
)
H.first_review_ts = (
    H.reviews[H.reviews['movie_slug'].isin(H.close_date_map)]
    .groupby('movie_slug')['estimated_timestamp'].min()
)
_new_first = H.first_review_ts.to_dict()
H.gaps['first_review_ts'] = H.gaps['slug'].map(_new_first)
H.gaps['gap_days'] = (
    H.gaps['close_ts'] - H.gaps['first_review_ts']
).dt.total_seconds() / 86400
H.gap_lookup = dict(zip(H.gaps['slug'], H.gaps['gap_days']))
print(f'noon-shift: {_n_shifted} day-level reviews shifted')


## Load tier 1 cache


In [ ]:
TIER1_CACHE = H.CACHE_DIR / 'phase1_ridge_tier1.pkl'
assert TIER1_CACHE.exists(), 'run phase1_ridge_tier1.ipynb first'
with open(TIER1_CACHE, 'rb') as f:
    df = pickle.load(f)
print(f'loaded {len(df)} rows  ·  columns: {len(df.columns)}')


## Configuration


In [ ]:
SNAP_DAYS_LIST = [5, 4, 3, 2, 1]
BASE_FEATURES = [
    'observed_count', 'first_review_dbc', 'target_gap', 'observed_rate',
    'rate_last_day', 'rate_first_day', 'top_critic_frac',
    'pub_diversity', 'pub_entropy', 'low_activity_frac',
]
TIER1_FEATURES = BASE_FEATURES + [
    'log_observed_count', 'log_rate_last_day', 'sqrt_rate_last_day', 'rate_delta',
]
TIER2_FEATURES = ['remaining_base_rate_sum', 'pool_mass_consumed', 'observed_top_tier_frac']
ALL_FEATURES = TIER1_FEATURES + TIER2_FEATURES

A1_POOL_SIZE = 20
TOP_TIER_N = 30
ALPHA_GRID = [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]
CV_FOLDS = 5
CV_SEED = 42
CACHE = H.CACHE_DIR / 'phase1_ridge_tier2.pkl'


## Build per-target A1 base_rate lookups

For each target, pick the 20 most recent resolved movies before target close (excluding target). Compute per-critic `base_rate[c] = movies_in_A1_reviewed_by_c / 20`. Cache as a dict keyed by target_slug → `{critic_name: base_rate}`. Also identify top-30 critics by base_rate within that A1 pool, and total base_rate sum.


In [ ]:
def build_a1_context(target_slug):
    close_ts = H.close_date_map[target_slug]
    training_slugs = H.default_training_slugs(
        H.movies, exclude_slug=target_slug,
        n=A1_POOL_SIZE, before_date=close_ts,
    )
    if len(training_slugs) < 5:
        return None
    train = H.reviews[H.reviews['movie_slug'].isin(training_slugs)]
    counts = train.groupby('reviewer_name')['movie_slug'].nunique()
    base_rate = (counts / A1_POOL_SIZE).to_dict()
    total_sum = float(sum(base_rate.values()))
    top_tier = set(counts.nlargest(TOP_TIER_N).index)
    return {
        'training_slugs': training_slugs,
        'base_rate': base_rate,
        'total_sum': total_sum,
        'top_tier': top_tier,
    }


a1_cache = {}
start = time.time()
for slug in sorted(H.close_date_map):
    ctx = build_a1_context(slug)
    if ctx is not None:
        a1_cache[slug] = ctx
print(f'built A1 pool for {len(a1_cache)} targets  ·  {time.time()-start:.1f}s')

# Sanity: typical A1 pool summary
sample_slug = list(a1_cache.keys())[0]
sample = a1_cache[sample_slug]
print(f'\nexample: {sample_slug}')
print(f'  training: {len(sample["training_slugs"])} movies')
print(f'  critics in pool: {len(sample["base_rate"])}')
print(f'  total_sum (mean reviews/movie proxy): {sample["total_sum"]:.2f}')
print(f'  top_tier size: {len(sample["top_tier"])}')
# Distribution of total_sum across targets
sums = [c['total_sum'] for c in a1_cache.values()]
print(f'  total_sum across targets: median={np.median(sums):.2f}  IQR={np.quantile(sums, 0.75) - np.quantile(sums, 0.25):.2f}')


## Compute finite-pool features per (target, snap)


In [ ]:
def compute_pool_features(target_slug, snap_days):
    ctx = a1_cache.get(target_slug)
    if ctx is None:
        return None
    close_ts = H.close_date_map[target_slug]
    midnight_utc = close_ts.floor('D')
    snap_time = midnight_utc - pd.Timedelta(days=snap_days)
    state = H.snapshot_state(target_slug, snap_time)
    if state is None:
        return None
    observed = state['observed_critics']

    base_rate = ctx['base_rate']
    total_sum = ctx['total_sum']
    top_tier = ctx['top_tier']

    obs_br_sum = float(sum(base_rate.get(c, 0.0) for c in observed))
    remaining_br_sum = max(total_sum - obs_br_sum, 0.0)
    pool_mass_consumed = (obs_br_sum / total_sum) if total_sum > 0 else 0.0
    top_observed = len(observed & top_tier)
    observed_top_tier_frac = top_observed / TOP_TIER_N

    return {
        'remaining_base_rate_sum': remaining_br_sum,
        'pool_mass_consumed': pool_mass_consumed,
        'observed_top_tier_frac': observed_top_tier_frac,
    }


# Populate tier-2 feature columns on df
for feat in TIER2_FEATURES:
    df[feat] = np.nan

start = time.time()
for idx, row in df.iterrows():
    feats = compute_pool_features(row['target_slug'], int(row['snap_days']))
    if feats is None:
        continue
    for k, v in feats.items():
        df.at[idx, k] = v
print(f'computed pool features for {len(df)} rows  ·  {time.time()-start:.1f}s')

# Quick distribution check
print('\nfeature distributions at T-3d:')
print(df[df['snap_days'] == 3][TIER2_FEATURES].describe().round(3).to_string())


## α CV per snap + LOO predict


In [ ]:
def select_alpha(X, y, standardize=True):
    kf = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=CV_SEED)
    best_alpha, best_mae = None, np.inf
    for alpha in ALPHA_GRID:
        fold_errs = []
        for train_idx, test_idx in kf.split(X):
            pipe = Pipeline([('scaler', StandardScaler()), ('ridge', Ridge(alpha=alpha))])
            pipe.fit(X[train_idx], y[train_idx])
            preds = pipe.predict(X[test_idx])
            fold_errs.extend(np.abs(preds - y[test_idx]).tolist())
        mae = float(np.mean(fold_errs))
        if mae < best_mae:
            best_mae, best_alpha = mae, alpha
    return best_alpha


def loo_predict(X, y, alpha):
    preds = np.zeros(len(X))
    for i in range(len(X)):
        mask = np.ones(len(X), dtype=bool)
        mask[i] = False
        pipe = Pipeline([('scaler', StandardScaler()), ('ridge', Ridge(alpha=alpha))])
        pipe.fit(X[mask], y[mask])
        preds[i] = pipe.predict(X[i:i+1])[0]
    return preds


snap_alpha_t2 = {}
df['ridge_t2_pred'] = np.nan
for snap_days in SNAP_DAYS_LIST:
    sub = df[df['snap_days'] == snap_days].dropna(subset=ALL_FEATURES + ['actual'])
    if len(sub) < CV_FOLDS * 2:
        continue
    X = sub[ALL_FEATURES].values
    y = sub['actual'].values.astype(float)
    best_alpha = select_alpha(X, y)
    snap_alpha_t2[snap_days] = best_alpha
    preds = loo_predict(X, y, best_alpha)
    df.loc[sub.index, 'ridge_t2_pred'] = preds
    print(f'  T-{snap_days}d (n={len(sub)}): α*={best_alpha}')

with open(CACHE, 'wb') as f:
    pickle.dump(df, f)
print(f'saved {CACHE}')


## Per-variant summary (5-way)


In [ ]:
def metrics(sub, pred_col):
    s = sub.dropna(subset=[pred_col])
    if len(s) == 0:
        return None
    err = s[pred_col].values - s['actual'].values
    abs_err = np.abs(err)
    return {
        'n': len(s), 'MAE': float(abs_err.mean()),
        'me': float(err.mean()),
        'p90_abs_err': float(np.quantile(abs_err, 0.9)),
    }


VARIANTS = [
    ('library',   'lib_pred'),
    ('ship',      'ship_pred'),
    ('ridge_orig','ridge_pred'),
    ('ridge_t1',  'ridge_t1_pred'),
    ('ridge_t2',  'ridge_t2_pred'),
]

print('=== cohort summary ===\n')
for snap_days in SNAP_DAYS_LIST:
    sub = df[df['snap_days'] == snap_days]
    print(f'T-{snap_days}d')
    for name, col in VARIANTS:
        m = metrics(sub, col)
        if m:
            print(f'  {name:12s}  n={m["n"]:3d}  MAE={m["MAE"]:6.2f}  '
                  f'me={m["me"]:+6.2f}  p90|e|={m["p90_abs_err"]:6.2f}')
    print()


## h/m subset


In [ ]:
HM = ['the_drama', 'the_super_mario_galaxy_movie', 'forbidden_fruits_2026',
      'they_will_kill_you', 'you_me_and_tuscany']
hm_df = df[df['target_slug'].isin(HM)]

print('=== h/m subset ===\n')
for snap_days in SNAP_DAYS_LIST:
    sub = hm_df[hm_df['snap_days'] == snap_days]
    if sub.empty:
        continue
    print(f'T-{snap_days}d (n={len(sub)})')
    for name, col in VARIANTS:
        m = metrics(sub, col)
        if m:
            print(f'  {name:12s}  MAE={m["MAE"]:6.2f}  me={m["me"]:+6.2f}')
    print()


## Paired bootstrap (tier 2 vs each baseline)


In [ ]:
COL_MAP = {name: col for name, col in VARIANTS}
PAIRS = [('ship', 'ridge_t2'), ('ridge_orig', 'ridge_t2'), ('ridge_t1', 'ridge_t2')]

print('=== paired bootstrap ΔMAE (A − B, +ve → B wins) ===\n')
print(f'{"snap":<6}{"A":<12}{"B":<12}{"Δ":>10}{"CI95_lo":>10}{"CI95_hi":>10}{"Δ %":>8}{"n":>6}  result')
for snap_days in SNAP_DAYS_LIST:
    snap_df = df[df['snap_days'] == snap_days]
    for a, b in PAIRS:
        a_col, b_col = COL_MAP[a], COL_MAP[b]
        paired = snap_df.dropna(subset=[a_col, b_col, 'actual'])
        if len(paired) < 5:
            continue
        a_abs = np.abs(paired[a_col].values - paired['actual'].values)
        b_abs = np.abs(paired[b_col].values - paired['actual'].values)
        deltas = a_abs - b_abs
        point, lo, hi = H.bootstrap_mae_delta(deltas, n_boot=1000)
        pct = 100 * point / a_abs.mean() if a_abs.mean() else float('nan')
        if lo > 0:
            result = f'{b:>12} wins'
        elif hi < 0:
            result = f'{a:>12} wins'
        else:
            result = '    ns'
        print(f'T-{snap_days}d  {a:<12}{b:<12}{point:>+10.3f}{lo:>+10.3f}{hi:>+10.3f}{pct:>+8.2f}{len(paired):>6}  {result}')
    print()


## Learned coefficients — does Ridge actually use finite-pool features?

If the tier-2 feature coefficients are near zero, finite-pool info is dead weight. If they're non-trivial, Ridge is learning something from pool composition.


In [ ]:
print('=== tier-2 Ridge coefficients (standardized scale) ===\n')
for snap_days in SNAP_DAYS_LIST:
    alpha = snap_alpha_t2.get(snap_days)
    if alpha is None:
        continue
    sub = df[df['snap_days'] == snap_days].dropna(subset=ALL_FEATURES + ['actual'])
    if len(sub) < 10:
        continue
    X = sub[ALL_FEATURES].values
    y = sub['actual'].values.astype(float)
    pipe = Pipeline([('scaler', StandardScaler()), ('ridge', Ridge(alpha=alpha))])
    pipe.fit(X, y)
    coefs = pipe.named_steps['ridge'].coef_
    intercept = pipe.named_steps['ridge'].intercept_
    pairs = sorted(zip(ALL_FEATURES, coefs), key=lambda p: -abs(p[1]))
    print(f'T-{snap_days}d  α={alpha}  intercept={intercept:.2f}')
    for feat, c in pairs:
        marker = '  ← pool' if feat in TIER2_FEATURES else ''
        print(f'  {feat:28s}  {c:+7.2f}{marker}')
    print()


## Plot: MAE by snap, all 5 variants


In [ ]:
import matplotlib.pyplot as plt

summary_rows = []
for snap_days in SNAP_DAYS_LIST:
    sub = df[df['snap_days'] == snap_days]
    for name, col in VARIANTS:
        m = metrics(sub, col)
        if m:
            summary_rows.append({'snap_days': snap_days, 'variant': name, **m})
summary = pd.DataFrame(summary_rows)

fig, ax = plt.subplots(1, 1, figsize=(10, 4.5))
colors = {'library': 'tab:gray', 'ship': 'tab:red',
          'ridge_orig': 'tab:blue', 'ridge_t1': 'tab:green', 'ridge_t2': 'tab:purple'}
markers = {'library': 's', 'ship': 'o', 'ridge_orig': '^',
           'ridge_t1': 'D', 'ridge_t2': '*'}
for name, _ in VARIANTS:
    sub = summary[summary['variant'] == name].sort_values('snap_days', ascending=False)
    ax.plot(sub['snap_days'], sub['MAE'], '-',
            marker=markers[name], color=colors[name], label=name, markersize=9)
ax.invert_xaxis()
ax.set_xlabel('snap days before close')
ax.set_ylabel('MAE (reviews)')
ax.set_title('Phase-1 MAE — tier-2 Ridge (+ finite-pool features) vs baselines')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## Observations

*(fill in after run)*
